# SymboLM Empirical Audit: Real 4-Arm Ablation & Hardware Valuation Suite

**Principal Systems Architect & Research Director:** **Sanish Gyawali** (Rupandehi, Nepal)  
**AI Systems Collaborator:** Antigravity (Google DeepMind)  
**Adversarial Scientific Protocol:** Claude (Haiku 5.5, Anthropic)  
**Target Model:** `HuggingFaceTB/SmolLM2-1.7B-Instruct` + `gyawalisanish0/SymboLM-2.1-SmolLM2-1.7B`  
**Hardware Target:** Kaggle Cloud NVIDIA Tesla T4 GPU  
**Pre-Registration Reference:** `PREREG_VALUATION.md`  

### Core Execution Phases:
1. **Task 2 Real 4-Arm Ablation (`eval/ablation_real.py`):**
   - Arm A: Step-by-Step CoT
   - Arm B: Answer-Only
   - Arm C: Canonical Register v2.1
   - Arm D: Shuffled/Corrupted Plan (Causal Intervention)
   - Evaluated across 200 clean held-out problems with Wilson 95% CIs and exact token counts.

2. **Hardware Valuation Protocol (`eval/power_profiler.py`):**
   - Physical energy sampling via `nvidia-smi` at 100ms intervals.
   - Measured baseline idle power subtraction.
   - 5 valuation arms including Arm D router-only control.
   - Paired bootstrap 95% confidence intervals on Joules per correct answer ($J/\text{correct}$).

In [ ]:
# Cell 1: Environment & GPU Verification
import os
import sys
import time
import torch

print('Python Version:', sys.version)
print('PyTorch Version:', torch.__version__)
if torch.cuda.is_available():
    print(f'[OK] CUDA Detected! Count: {torch.cuda.device_count()}')
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f'  GPU {i}: {props.name} ({props.total_memory / 1e9:.2f} GB VRAM)')
    !nvidia-smi
else:
    print('[WARNING] CUDA not detected!')


In [ ]:
# Cell 2: Dependencies Installation
!pip install -q --upgrade pip
!pip uninstall -y -q torchao
!pip install -q 'transformers>=4.48.0' 'peft>=0.14.0' 'accelerate>=1.2.0' 'sympy>=1.12' 'datasets>=3.2.0'


In [ ]:
# Cell 3: Clone Audit Branch from GitHub
!rm -rf /kaggle/working/symboLM
!git clone -b audit/empirical-hardening https://github.com/gyawalisanish0/symboLM.git /kaggle/working/symboLM
%cd /kaggle/working/symboLM
!git log -n 3 --oneline


In [ ]:
# Cell 4: Fuse SymboLM SmolLM2 Weights for High-Throughput Inference
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

fused_dir = '/kaggle/working/symbolm_smollm_fused'
if not os.path.exists(fused_dir):
    print('Fusing base model and SymboLM v2.1 adapter...')
    base_model_id = 'HuggingFaceTB/SmolLM2-1.7B-Instruct'
    adapter_id = 'gyawalisanish0/SymboLM-2.1-SmolLM2-1.7B'
    
    tokenizer = AutoTokenizer.from_pretrained(adapter_id, trust_remote_code=True)
    base = AutoModelForCausalLM.from_pretrained(
        base_model_id,
        torch_dtype=torch.float16,
        device_map='auto',
        trust_remote_code=True,
    )
    model = PeftModel.from_pretrained(base, adapter_id)
    merged_model = model.merge_and_unload()
    
    merged_model.save_pretrained(fused_dir)
    tokenizer.save_pretrained(fused_dir)
    print(f'[OK] Standalone fused weights saved to: {fused_dir}')
else:
    print(f'[OK] Fused weights already exist at: {fused_dir}')


In [ ]:
# Cell 5: Execute Task 2 Real 4-Arm Ablation Suite
%cd /kaggle/working/symboLM
!python eval/ablation_real.py \
    --input data/held_out_eval_200_clean.jsonl \
    --model /kaggle/working/symbolm_smollm_fused \
    --output eval/ablation_real_results.json \
    --report eval/ABLATION_REAL_REPORT.md \
    --device cuda \
    --max-tokens 4096


In [ ]:
# Cell 6: Execute Hardware Valuation & Dynamic Energy Profiling Suite
%cd /kaggle/working/symboLM
!python eval/power_profiler.py \
    --input data/held_out_eval_200_clean.jsonl \
    --model /kaggle/working/symbolm_smollm_fused \
    --out-dir results/valuation \
    --report VALUATION.md \
    --device cuda \
    --max-tokens 4096


In [ ]:
# Cell 7: Display Scorecards and Package Tarball Artifacts
import os
from pathlib import Path

print('=' * 80)
print('TASK 2 ABLATION REPORT:')
print('=' * 80)
rep_ablation = Path('/kaggle/working/symboLM/eval/ABLATION_REAL_REPORT.md')
if rep_ablation.exists():
    print(rep_ablation.read_text(encoding='utf-8'))

print('=' * 80)
print('HARDWARE VALUATION REPORT:')
print('=' * 80)
rep_val = Path('/kaggle/working/symboLM/VALUATION.md')
if rep_val.exists():
    print(rep_val.read_text(encoding='utf-8'))

# Package artifacts
!tar -czvf /kaggle/working/symbolm_audit_and_valuation_receipts.tar.gz \
    -C /kaggle/working/symboLM \
    PREREG_VALUATION.md \
    VALUATION.md \
    eval/ablation_real_results.json \
    eval/ABLATION_REAL_REPORT.md \
    results/valuation/

print('[DONE] Receipts archived to /kaggle/working/symbolm_audit_and_valuation_receipts.tar.gz')
